
# Union Operations — `union()` and its variations
- Both on a DataFrame and using Spark SQL


## Covered union methods —
- `union()` vs `unionAll()` vs `unionByName()`

### Roadmap
1. Data setup — `emp_df` and `additional_emp_df`
2. Union fundamentals and the DataFrame vs Spark SQL comparison
3. DataFrame operations — `union()`, `unionAll()`, `union().distinct()`
4. Spark SQL operations — `UNION` vs `UNION ALL`
5. Duplicate data — what happens when appended rows already exist
6. `unionByName()` — name-based column matching
7. Column-count mismatch — `allowMissingColumns=True`

In [0]:
# Path to the employee CSV dataset
EMPLOYEE_DATA = "/Workspace/Repos/mayurkedari16@gmail.com/pyspark-databricks/data/employees-data/employees.csv"

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, column, expr, lit

# Create a SparkSession — the entry point for DataFrame and SQL operations
spark = SparkSession.builder.appName("aggregations").getOrCreate()

In [0]:
# Read the CSV with header and PERMISSIVE mode (bad rows become null instead of failing)
employee_df = (
    spark.read.format("csv")
              .option("header", True)
              .option("mode", "PERMISSIVE")
              .option("path", EMPLOYEE_DATA)
              .load()
)

employee_df.createOrReplaceTempView("employees_tbl")
employee_df.printSchema()
employee_df.show(20)


## Data Setup — `emp_df` and `additional_emp_df`

In [0]:
from pyspark.sql.functions import col, lit, expr, count
from pyspark.sql import functions as F

# Shared schema for the main employee DataFrame used across all union examples
schema_columns = ["emp_id", "name", "salary", "mgr_id"]

# Main DataFrame — 9 employees, the base that every union example below appends to
emp_data = [
    (10, 'Anil', 50000, 18),
    (11, 'Vikas', 75000, 16),
    (12, 'Nisha', 40000, 18),
    (13, 'Nidhi', 60000, 17),
    (14, 'Priya', 80000, 18),
    (15, 'Mohit', 45000, 18),
    (16, 'Rajesh', 90000, 10),
    (17, 'Raman', 55000, 16),
    (18, 'Sam', 65000, 17),
]

print("============================= Employee Data =============================")
emp_df = spark.createDataFrame(data=emp_data, schema=schema_columns)
print("Employee DF Count:", emp_df.count())
emp_df.printSchema()
emp_df.show(3)

print("============================= Additional 2 Employees Data =============================")
# New DataFrame — 2 extra employees that get appended to emp_df in the union examples
additional_emp_schema_columns = ["emp_id", "name", "salary", "mgr_id"]
additional_emp_data = [
    (19, 'Sohan', 50000, 18),
    (20, 'Sima', 75000, 17),
]
additional_emp_df = spark.createDataFrame(data=additional_emp_data, schema=additional_emp_schema_columns)
print("Additional Employees DF Count:", additional_emp_df.count())
additional_emp_df.printSchema()
additional_emp_df.show(3)





## Union — combining two DataFrames
- `union()` appends the rows of one DataFrame below another — matching columns by **position** (order), not by name
- `unionAll()` — it behaves identically to `union()` and serves as the explicit DataFrame counterpart of SQL's `UNION ALL` (it was briefly marked deprecated in Spark 2.x, but that was later reverted)
- Both **keep duplicates** (no dedup) and require the **same number of columns** in the same order
- The resulting column names come from the **left** DataFrame — only positions matter

## Notes — DataFrame API vs Spark SQL

> [!IMPORTANT]
> `union()` and `unionAll()` DataFrame methods behave exactly the same — but the same operation in Spark SQL differs: `UNION` deduplicates while `UNION ALL` keeps every row.

| Method | Duplicates | Column matching | Equivalent in the other API |
| --- | --- | --- | --- |
| `union()` (DataFrame) | kept | position | `UNION ALL` |
| `unionAll()` (DataFrame) | kept | position | `UNION ALL` |
| `UNION` (SQL) | removed (distinct) | position | `union().distinct()` |
| `UNION ALL` (SQL) | kept | position | `union()` |

- To reproduce SQL's `UNION` in the DataFrame API, chain `.distinct()` after `union()`
- ⚠️ Appending data that already exists: `union()` and `unionAll()` blindly keep the duplicate rows, while SQL `UNION` silently drops them (demo below)
- 💡 Summary — every DataFrame union method (`union()`, `unionAll()`, `unionByName()`) and Spark SQL `UNION ALL` produce the **same stacked output, duplicates included**; only `UNION` in Spark SQL differs by excluding duplicates as you append (it is shorthand for `UNION DISTINCT`)

#### DataFrame Operations - union() vs unionAll()

In [0]:
# union() stacks rows by column POSITION (not name) — appends additional_emp_df below emp_df
results = emp_df.union(additional_emp_df)
# Count: 9 + 2 = 11 rows — union() keeps duplicates, it does NOT deduplicate
print("After union() Count:", results.count())
# Column names come from the left DataFrame — only positions matter
results.show(30)

In [0]:
# unionAll() is the explicit DataFrame counterpart of SQL UNION ALL — identical position-based stack to union(), keeps duplicates
results = emp_df.unionAll(additional_emp_df)
# Count: same 11 rows as union() — unionAll() differs only in name, not behavior
print("After unionAll() Count:", results.count())
results.show(30)

In [0]:
# union().distinct() — the DataFrame equivalent of SQL UNION: stack rows first, then drop duplicates
results = emp_df.union(additional_emp_df).distinct()
# Count: 11 — matches SQL UNION, because emp_df and additional_emp_df share no identical rows
print("After union().distinct() Count:", results.count())
results.show(30)

# Proof of the dedup — unioning a DataFrame with ITSELF doubles the rows, and distinct() collapses them back
print("emp_df.union(emp_df) Count:", emp_df.union(emp_df).count())
print("emp_df.union(emp_df).distinct() Count:", emp_df.union(emp_df).distinct().count())


#### Spark SQL Operations - union() vs unionAll()

In [0]:
# Register both DataFrames as temp views so Spark SQL can query them
emp_df.createOrReplaceTempView("emp_df_tbl")
additional_emp_df.createOrReplaceTempView("additional_emp_df_tbl")

print("================== UNION ======================")
# SQL UNION deduplicates — equivalent to DataFrame union() followed by distinct()
results = spark.sql("""
    SELECT * FROM emp_df_tbl
    UNION
    SELECT * FROM additional_emp_df_tbl
""")
# Count stays 11 only because the two sides share no identical rows — UNION removes duplicates
print("UNION Count using Spark SQL:", results.count())
results.show(30)

print("================== UNION ALL ======================")
# SQL UNION ALL keeps every row — the true SQL equivalent of DataFrame union()
results = spark.sql("""
    SELECT * FROM emp_df_tbl
    UNION ALL
    SELECT * FROM additional_emp_df_tbl
""")
print("UNION ALL Count using Spark SQL:", results.count())
results.show(30)

#### Duplicate data — appended rows that already exist

- `union()` and `unionAll()` blindly stack rows — appending data that is already present leaves the duplicates in (the count grows)
- SQL `UNION` silently drops those duplicates again, because it applies `distinct` semantics
- ⚠️ SQL `UNION ALL` behaves exactly like the DataFrame `union()` — the duplicates stay

In [0]:
# Duplicate rows — emp_ids 10 and 11 already exist in emp_df, so this appended data is fully redundant
additional_duplicate_data = [
    (10, 'Anil', 50000, 18),
    (11, 'Vikas', 75000, 16),
]
additional_duplicate_df = spark.createDataFrame(data=additional_duplicate_data, schema=schema_columns)

# union() with duplicate data — count grows to 11: the redundant rows are blindly stacked
print("union() Count with duplicate data:", emp_df.union(additional_duplicate_df).count())

# unionAll() with duplicate data — also 11: identical behavior, duplicates kept
print("unionAll() Count with duplicate data:", emp_df.unionAll(additional_duplicate_df).count())

# Register both sides as temp views so Spark SQL can run the same comparison
emp_df.createOrReplaceTempView("emp_df_tbl")
additional_duplicate_df.createOrReplaceTempView("additional_duplicate_df_tbl")

# SQL UNION [DISTINCT] — count stays 9: the duplicated rows are removed because UNION deduplicates
results = spark.sql("""
    SELECT * FROM emp_df_tbl
    UNION
    SELECT * FROM additional_duplicate_df_tbl
""")
print("SQL UNION [DISTINCT] Count without duplicate data:", results.count())
results.show(30)

# SQL UNION ALL — count grows to 11: the true SQL equivalent of DataFrame union()/unionAll()
results = spark.sql("""
    SELECT * FROM emp_df_tbl
    UNION ALL
    SELECT * FROM additional_duplicate_df_tbl
""")
print("SQL UNION ALL Count with duplicate data:", results.count())
results.show(30)

#### `unionByName()` — name-based column matching

- Matches columns by **name** instead of position — the fix when both DataFrames hold the same columns in a different order
- Still expects the **same set of columns** — an extra or missing column fails with `NUM_COLUMNS_MISMATCH`
- 💡 `unionByName(..., allowMissingColumns=True)` fills columns that exist on only one side with `null`
- There is **no Spark SQL equivalent** — in SQL you must manually `SELECT` the columns into the same order before `UNION`

In [0]:
# Test rows whose column ORDER differs from emp_df — values arrive as emp_id, salary, mgr_id, name
additional_wrong_column_data = [
    (19, 50000, 18, 'Sohan'),
    (20, 75000, 17, 'Sima'),
]
# Schema declares that shifted order explicitly
additional_wrong_column_data_schema = ["emp_id", "salary", "mgr_id", "name"]

additional_wrong_column_data_df = spark.createDataFrame(data=additional_wrong_column_data, schema=additional_wrong_column_data_schema)

# union() is position-based — it would silently append salary into the name column and mgr_id into salary (commented out to avoid the corruption)
# results = emp_df.union(additional_wrong_column_data_df)
# print("Count:", results.count())
# results.show(30)
# results.printSchema()

# unionByName() matches columns by NAME instead of position — the correct tool when both sides have the same columns in a different order
results = emp_df.unionByName(additional_wrong_column_data_df)
# Count: 11 rows with every value landing in the right column despite the different column order
print("Count:", results.count())
results.show(30)
results.printSchema()

#### Column-count mismatch — extra columns

- `union()`, `unionAll()` and `unionByName()` all fail with `NUM_COLUMNS_MISMATCH` when the inputs have a different number of columns
- 💡 Only `unionByName(..., allowMissingColumns=True)` survives it — unmatched columns are padded with `null`

In [0]:
# Test rows carrying an EXTRA column — a column-count mismatch no plain union variation can reconcile
additional_wrong_with_extra_column_data = [
    (19, 50000, 18, 'Sohan', 10),
    (20, 75000, 17, 'Sima', 20),
]
# Schema now has 5 columns vs emp_df's 4
additional_wrong_with_extra_column_data_schema = ["emp_id", "salary", "mgr_id", "name", "extra_col"]

additional_wrong_with_extra_column_data_df = spark.createDataFrame(additional_wrong_with_extra_column_data, additional_wrong_with_extra_column_data_schema)

# All three fail with NUM_COLUMNS_MISMATCH — "first input has 4 columns and the second input has 5 columns" (kept commented to avoid the error)
# results = emp_df.union(additional_wrong_with_extra_column_data_df)        # union() — fails
# results = emp_df.unionAll(additional_wrong_with_extra_column_data_df)     # unionAll() — fails
# results = emp_df.unionByName(additional_wrong_with_extra_column_data_df)  # unionByName() — still fails

# allowMissingColumns=True is the ONLY escape — columns present on just one side (like extra_col) are padded with null
results = emp_df.unionByName(additional_wrong_with_extra_column_data_df, allowMissingColumns=True)

# Count: 11 rows — emp_df's side simply shows null for extra_col
print("Count:", results.count())
results.show(30)
results.printSchema()
